In [4]:
# conda_lib <- "/data/ag_reichart_lindberg/nhadizad/miniconda3/envs/old_R_433_for_cellchat_161/lib/R/library"
# .libPaths(conda_lib)
# install.packages("readr", repos = "https://cloud.r-project.org")

installiere auch Abhängigkeiten ‘bit’, ‘bit64’, ‘progress’, ‘hms’, ‘vroom’, ‘tzdb’


Aktualisiere HTML-Index der Pakete in '.Library'

Making 'packages.html' ...
 fertig



In [1]:
# ============================================================
# Setup
# ============================================================
conda_lib <- "/data/ag_reichart_lindberg/nhadizad/miniconda3/envs/old_R_433_for_cellchat_161/lib/R/library"
.libPaths(conda_lib)

suppressMessages({
  library(CellChat)
  library(ggplot2)
  library(patchwork)
  library(dplyr)
  library(tidyr)
  library(reshape2)
  library(scales)
  library(circlize)
  # library(readr)
  library(stringr)
  library(future)
})
options(stringsAsFactors = FALSE)
options(future.globals.maxSize = 10 * 1024^3)
plan("multicore", workers = 15)

# --- Global constants (edit ONLY here) ---
COND_COLORS  <- c("Rejection" = "#E09E9E", "3month-protocol" = "skyblue")
COND_ORDER   <- c("Rejection", "3month-protocol")

BASE_OUT <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results"

safe_name <- function(x) gsub("[/\\\\ ]", "_", x)

make_dir <- function(...) {
  d <- file.path(...)
  if (!dir.exists(d)) dir.create(d, recursive = TRUE)
  d
}
# Plotting details & parameters
save_gg <- function(plot, subdir, filename, w = 7, h = 7) {
  d <- make_dir(BASE_OUT, subdir)
  f <- file.path(d, filename)
  ggsave(f, plot = plot, device = "pdf",
         width = w, height = h, units = "in",
         dpi = 300, useDingbats = FALSE)
  cat("Saved:", f, "\n")
  invisible(f)
}

# save_csv <- function(df, subdir, filename) {
#   d <- make_dir(BASE_OUT, subdir)
#   f <- file.path(d, filename)
#   write_csv(df, f)
#   cat("Saved:", f, "\n")
#   invisible(f)
# }
save_csv <- function(df, subdir, filename) {
  d <- make_dir(BASE_OUT, subdir)
  f <- file.path(d, filename)
  # row.names = FALSE keeps the CSV clean like readr's format
  write.csv(df, f, row.names = FALSE) 
  cat("Saved:", f, "\n")
  invisible(f)
}


# Checking to ensure we are using log normalized data

In [15]:
# Check that createCellChat actually used the data layer
cat("=== Data layer used by CellChat ===\n")

# Re-create one object temporarily to verify
seurat_test <- readRDS("/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/creating_seurat_obj/objects_seurat_fractions_and_whole_UMI_merged/seurat_fraction_Rejection.rds")

# Check what createCellChat will extract
# In CellChat 1.6.1 with Seurat v5 Assay5, we must extract manually
data_mat <- GetAssayData(seurat_test, assay = "RNA", layer = "data")

cat("Max value in data layer:", max(data_mat), "\n")
cat("Mean non-zero value:", mean(data_mat@x), "\n")

# Now check what is actually inside the saved CellChat object
cc_test <- readRDS("/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/creating_cellchat_obj/objects_cellchat_fractions/CellChat_obj_Rejection_netP.RDS")

cat("\n=== Max value inside cc_rej@data.signaling ===\n")
print(max(cc_test@data.signaling))

cat("\n=== Mean non-zero value inside cc_rej@data.signaling ===\n")
print(mean(cc_test@data.signaling[cc_test@data.signaling > 0]))

cat("\n=== Dimensions of data.signaling ===\n")
print(dim(cc_test@data.signaling))

cat("\n=== Number of cells ===\n")
print(ncol(cc_test@data.signaling))

cat("\n=== Number of cells in old Rejection (if you have it) ===\n")
# Only run this if you still have the old object accessible
# cc_old <- readRDS("path/to/old/CellChat_obj_rejection_netP.RDS")
# print(ncol(cc_old@data.signaling))

invisible(NULL)

=== Data layer used by CellChat ===
Max value in data layer: 8.23502 
Mean non-zero value: 1.704327 

=== Max value inside cc_rej@data.signaling ===
[1] 6.733793

=== Mean non-zero value inside cc_rej@data.signaling ===
[1] 1.906236

=== Dimensions of data.signaling ===
[1]   931 49482

=== Number of cells ===
[1] 49482

=== Number of cells in old Rejection (if you have it) ===


In [13]:
# Restart R kernel completely first, then run ONLY this diagnostic
# Do not load any other libraries before this

cat("=== SeuratObject version ===\n")
print(packageVersion("SeuratObject"))

cat("\n=== Seurat version ===\n")
print(packageVersion("Seurat"))

cat("\n=== CellChat version ===\n")
print(packageVersion("CellChat"))

cat("\n=== Search path (loaded packages) ===\n")
print(search())

cat("\n=== .libPaths() ===\n")
print(.libPaths())

invisible(NULL)

=== Object class ===
[1] "Seurat"
attr(,"package")
[1] "SeuratObject"

=== Assay names ===
[1] "RNA"

=== RNA assay class ===
[1] "Assay5"
attr(,"package")
[1] "SeuratObject"

=== Slot names of RNA assay ===
[1] "layers"     "cells"      "features"   "default"    "assay.orig"
[6] "meta.data"  "misc"       "key"       

=== Max value in data matrix ===
[1] 8.23502
=== Min value ===
[1] 0
=== First 3x3 ===
3 x 3 sparse Matrix of class "dgCMatrix"
           PAV76_GACCTTCTCATCCTAT-1 PAV76_TTCCGTGCACTCTAGA-1
5S-rRNA                           .                        .
5S-rRNA-1                         .                        .
5S-rRNA-10                        .                        .
           PAV76_GTCACTCCAACGCCCA-1
5S-rRNA                           .
5S-rRNA-1                         .
5S-rRNA-10                        .


# Load Objects and Merge

In [2]:
# ============================================================
# SECTION 1: Load Objects and Merge
# ============================================================

RDS_DIR <- "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/creating_cellchat_obj/objects_cellchat_fractions"

cc_rej   <- readRDS(file.path(RDS_DIR, "CellChat_obj_Rejection_netP.RDS"))
cc_3m    <- readRDS(file.path(RDS_DIR, "CellChat_obj_3month-protocol_netP.RDS"))

# Align cell type universe across both objects
group.new <- union(levels(cc_rej@idents), levels(cc_3m@idents))
cc_rej <- liftCellChat(cc_rej, group.new)
cc_3m  <- liftCellChat(cc_3m,  group.new)

# Merge — Rejection FIRST to match COND_ORDER
object.list   <- list(Rejection = cc_rej, `3month-protocol` = cc_3m)
cellchat_merged <- mergeCellChat(object.list,
                                 add.names = names(object.list),
                                 cell.prefix = TRUE)

cat("Merged object ready. Conditions:", paste(names(object.list), collapse = ", "), "\n")
cat("Cell types:", paste(group.new, collapse = ", "), "\n")



The CellChat object will be lifted up using the cell labels CM, EC, EC_papio, FB, Lymph_papio, Lymph_sus, MC, Mye_papio, Mye_sus, NC



Update slots object@net, object@netP, object@idents in a single dataset... 


The CellChat object will be lifted up using the cell labels CM, EC, EC_papio, FB, Lymph_papio, Lymph_sus, MC, Mye_papio, Mye_sus, NC



Update slots object@net, object@netP, object@idents in a single dataset... 


Warning message in mergeCellChat(object.list, add.names = names(object.list), cell.prefix = TRUE):
“Prefix cell names!”


The cell barcodes in merged 'meta' is  PAV76_GACCTTCTCATCCTAT-1 PAV76_TTCCGTGCACTCTAGA-1 PAV76_GTCACTCCAACGCCCA-1 PAV76_CCGTTCAGTACGCTAT-1 PAV76_TCCCAGTAGTAGAATC-1 PAV76_TCCTCTTCACACCTTC-1 


Warning message in mergeCellChat(object.list, add.names = names(object.list), cell.prefix = TRUE):
“The cell barcodes in merged 'meta' is different from those in the used data matrix.
              We now simply assign the colnames in the data matrix to the rownames of merged 'mata'!”


Merge the following slots: 'data.signaling','images','net', 'netP','meta', 'idents', 'var.features' , 'DB', and 'LR'.



Merged object ready. Conditions: Rejection, 3month-protocol 


Cell types: CM, EC, EC_papio, FB, Lymph_papio, Lymph_sus, MC, Mye_papio, Mye_sus, NC 


# Raw Interaction Tables (CSVs)

In [3]:
# ============================================================
# SECTION 2: Raw Interaction Tables (CSVs)
# ============================================================

df_rej <- subsetCommunication(cc_rej)
df_3m  <- subsetCommunication(cc_3m)

save_csv(df_rej, "01_tables", "CellChat_Rejection_all_interactions.csv")
save_csv(df_3m,  "01_tables", "CellChat_3month_all_interactions.csv")

# Wide comparison table per pathway
ranknet_raw <- rankNet(
  cellchat_merged, mode = "comparison",
  stacked = FALSE, do.stat = TRUE,
  cutoff.pvalue = 1,          # keep ALL pathways
  return.data = TRUE, measure = "weight"
)$signaling.contribution

save_csv(ranknet_raw, "01_tables", "RankNet_all_pathways_long.csv")

wide_ranknet <- ranknet_raw %>%
  select(name, contribution, pvalues, group) %>%
  pivot_wider(id_cols = name,
              names_from = group,
              values_from = c(contribution, pvalues))
save_csv(wide_ranknet, "01_tables", "RankNet_all_pathways_wide.csv")

# ── ADDED 6 Oct 2026: which rankNet P values are real tests, and BH across those ────────────────────────────
# rankNet's P is a paired Wilcoxon test over the sender-receiver cell-type pairs of one pathway (the pairs, not the
# animals, are the units). CellChat writes P = 0 - which is NOT a test - whenever it cannot run the test (pathway absent
# in one condition, 3 or fewer non-zero pairs, or an undefined result); a real Wilcoxon P is never exactly 0, so those
# rows get test_performed = FALSE and NA. fdr_BH = Benjamini-Hochberg across the pathways actually tested here.
.pcol <- grep("^pvalues_", names(wide_ranknet), value = TRUE)[1]
wide_ranknet$test_performed <- !is.na(wide_ranknet[[.pcol]]) & wide_ranknet[[.pcol]] > 0
wide_ranknet$p_wilcox_pairs <- ifelse(wide_ranknet$test_performed, wide_ranknet[[.pcol]], NA)
wide_ranknet$fdr_BH <- NA_real_
wide_ranknet$fdr_BH[wide_ranknet$test_performed] <- p.adjust(wide_ranknet$p_wilcox_pairs[wide_ranknet$test_performed], method = "BH")
save_csv(wide_ranknet, "01_tables", "RankNet_all_pathways_wide.csv")
cat(sum(wide_ranknet$test_performed), "of", nrow(wide_ranknet), "pathways tested;",
    sum(wide_ranknet$fdr_BH < 0.05, na.rm = TRUE), "with BH < 0.05\n")
print(as.data.frame(wide_ranknet[wide_ranknet$test_performed, c("name", "p_wilcox_pairs", "fdr_BH")][order(wide_ranknet$fdr_BH[wide_ranknet$test_performed]), ]), row.names = FALSE)


Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/01_tables/CellChat_Rejection_all_interactions.csv 


Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/01_tables/CellChat_3month_all_interactions.csv 


Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/01_tables/RankNet_all_pathways_long.csv 


Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/01_tables/RankNet_all_pathways_wide.csv 


Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/01_tables/RankNet_all_pathways_wide.csv 


20 of 41 pathways tested; 7 with BH < 0.05


     name p_wilcox_pairs       fdr_BH
  LAMININ   1.855162e-05 0.0001855162
 COLLAGEN   1.480755e-05 0.0001855162
      FN1   1.220703e-04 0.0008138021
     EPHA   3.051758e-04 0.0015258789
      BMP   1.953125e-03 0.0078125000
    PTPRM   4.082921e-03 0.0136097359
     CADM   8.969327e-03 0.0256266497
    SEMA3   2.453613e-02 0.0613403320
   PECAM1   5.736434e-02 0.1274763061
      APP   1.093750e-01 0.2187500000
     PDGF   2.500000e-01 0.4166666667
      CSF   2.500000e-01 0.4166666667
     THBS   4.238281e-01 0.6145833333
    SEMA6   4.375000e-01 0.6145833333
     NCAM   4.609375e-01 0.6145833333
      FGF   8.984375e-01 0.9457236842
    NOTCH   8.457031e-01 0.9457236842
      IGF   8.393555e-01 0.9457236842
      JAM   8.539233e-01 0.9457236842
     CD45   1.000000e+00 1.0000000000


# Global Interaction Overview

In [4]:
# ============================================================
# SECTION 3: Global Interaction Overview (CORRECTED v5)
# ============================================================

sub3 <- make_dir(BASE_OUT, "02_global_overview")

# --- Helper: save base-R plots to PDF without Jupyter interference ---
save_base_plot <- function(filepath, width = 7, height = 7, plot_expr) {
  while (dev.cur() > 1) dev.off()
  pdf(filepath, width = width, height = height)
  tryCatch(
    force(plot_expr),
    error = function(e) {
      plot.new()
      title(paste("Plot failed:", e$message), cex.main = 0.8)
    }
  )
  dev.off()
  while (dev.cur() > 1) dev.off()
  cat("Saved:", filepath, "\n")
  invisible(NULL)
}

# ----------------------------------------------------------------
## 3a. Bar charts — colors directly mapped to Condition Names
# ----------------------------------------------------------------
gg_count_raw  <- compareInteractions(cellchat_merged, show.legend = TRUE,
                                     group = c(1, 2), measure = "count")
gg_weight_raw <- compareInteractions(cellchat_merged, show.legend = TRUE,
                                     group = c(1, 2), measure = "weight")

remap_colors <- function(gg_raw, title_label, y_label) {
  gg_raw +
    # 1. Manually set the fill colors using the exact condition names
    scale_fill_manual(
      values = c("Rejection" = "#E09E9E", "3month-protocol" = "skyblue")
    ) +
    # 2. Use a DISCRETE scale since the values are text strings, not numbers
    scale_x_discrete(
      limits = c("Rejection", "3month-protocol")
    ) +
    labs(title = title_label, x = NULL, y = y_label, fill = "Condition") +
    theme_classic(base_size = 14) +
    theme(plot.title      = element_text(hjust = 0.5, face = "bold"),
          legend.position = "top")
}

gg_count  <- remap_colors(gg_count_raw,  "Number of Interactions", "Count")
gg_weight <- remap_colors(gg_weight_raw, "Interaction Strength",   "Total Weight")
combined_global <- gg_count + gg_weight + plot_layout(ncol = 2)

ggsave(
  filename = file.path(sub3, "Global_interactions_count_and_weight.pdf"),
  plot     = combined_global,
  device   = "pdf",
  width = 8, height = 5, units = "in",
  dpi = 300, useDingbats = FALSE
)
cat("Saved: Global_interactions_count_and_weight.pdf\n")
invisible(NULL)

# ----------------------------------------------------------------
## 3b. Circle plots
# ----------------------------------------------------------------
weight_max_count  <- getMaxWeight(object.list, attribute = "count")
weight_max_weight <- getMaxWeight(object.list, attribute = "weight")

save_base_plot(
  file.path(sub3, "DiffInteraction_circle_count.pdf"),
  width = 8, height = 8,
  plot_expr = {
    par(mfrow = c(1, 2), mar = c(1, 1, 4, 1), oma = c(0, 0, 2, 0))
    for (i in seq_along(object.list)) {
      netVisual_circle(
        object.list[[i]]@net$count,
        weight.scale    = TRUE,
        edge.weight.max = weight_max_count,
        edge.width.max  = 10,
        title.name      = names(object.list)[i]
      )
    }
    mtext("Number of Interactions per Condition",
          outer = TRUE, cex = 1.2, font = 2, line = 0.5)
  }
)

save_base_plot(
  file.path(sub3, "DiffInteraction_circle_weight.pdf"),
  width = 8, height = 8,
  plot_expr = {
    par(mfrow = c(1, 2), mar = c(1, 1, 4, 1), oma = c(0, 0, 2, 0))
    for (i in seq_along(object.list)) {
      netVisual_circle(
        object.list[[i]]@net$weight,
        weight.scale    = TRUE,
        edge.weight.max = weight_max_weight,
        edge.width.max  = 10,
        title.name      = names(object.list)[i]
      )
    }
    mtext("Interaction Strength per Condition",
          outer = TRUE, cex = 1.2, font = 2, line = 0.5)
  }
)

# ----------------------------------------------------------------
## 3c. Heatmaps — called on each individual object
# ----------------------------------------------------------------

# ----------------------------------------------------------------
## 3c. Heatmaps — explicitly using print() for ComplexHeatmap rendering
# ----------------------------------------------------------------

# Side-by-side count heatmaps
save_base_plot(
  file.path(sub3, "Heatmap_count_Rejection.pdf"),
  width = 8, height = 7,
  plot_expr = {
    par(mar = c(4, 4, 5, 4))
    # FIX: Wrapped in print() to force rendering inside the pdf() device
    print(netVisual_heatmap(cc_rej, measure = "count",
                      title.name = "Rejection — interaction count"))
  }
)

save_base_plot(
  file.path(sub3, "Heatmap_count_3month.pdf"),
  width = 8, height = 7,
  plot_expr = {
    par(mar = c(4, 4, 5, 4))
    print(netVisual_heatmap(cc_3m, measure = "count",
                      title.name = "3month-protocol — interaction count"))
  }
)

# Side-by-side weight heatmaps
save_base_plot(
  file.path(sub3, "Heatmap_weight_Rejection.pdf"),
  width = 8, height = 7,
  plot_expr = {
    par(mar = c(4, 4, 5, 4))
    print(netVisual_heatmap(cc_rej, measure = "weight",
                      title.name = "Rejection — interaction strength"))
  }
)

save_base_plot(
  file.path(sub3, "Heatmap_weight_3month.pdf"),
  width = 8, height = 7,
  plot_expr = {
    par(mar = c(4, 4, 5, 4))
    print(netVisual_heatmap(cc_3m, measure = "weight",
                      title.name = "3month-protocol — interaction strength"))
  }
)

cat("Section 3 complete.\n")
invisible(NULL)

Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.
Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.


Saved: Global_interactions_count_and_weight.pdf
Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/02_global_overview/DiffInteraction_circle_count.pdf 
Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/02_global_overview/DiffInteraction_circle_weight.pdf 


Do heatmap based on a single object 




Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/02_global_overview/Heatmap_count_Rejection.pdf 


Do heatmap based on a single object 




Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/02_global_overview/Heatmap_count_3month.pdf 


Do heatmap based on a single object 




Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/02_global_overview/Heatmap_weight_Rejection.pdf 


Do heatmap based on a single object 




Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/02_global_overview/Heatmap_weight_3month.pdf 
Section 3 complete.


# RankNet — Differential Signaling Pathways

In [5]:
# ============================================================
# SECTION 4: RankNet — Differential Signaling Pathways
# (Main Figure candidate: panel B)
# ============================================================

sub4 <- make_dir(BASE_OUT, "03_rankNet")

# Define explicit mapping to prevent any factor-level swapping
explicit_colors <- c("Rejection" = "#E09E9E", "3month-protocol" = "skyblue")

## 4a. Stacked (relative contribution per condition)
p_stacked <- rankNet(
  cellchat_merged, mode = "comparison",
  stacked = TRUE, do.stat = TRUE
) +
  scale_fill_manual(
    breaks = c("Rejection", "3month-protocol"), # Locks legend order
    values = explicit_colors                    # Locks exact colors to names
  ) +
  labs(fill = "Condition") +
  theme_classic(base_size = 12) +
  theme(panel.grid = element_blank(),
        axis.text.y = element_text(size = 9))

save_gg(p_stacked, "03_rankNet",
        "RankNet_stacked_contribution.pdf", w = 6, h = 8)

## 4b. Unstacked (side-by-side absolute contribution)
p_unstacked <- rankNet(
  cellchat_merged, mode = "comparison",
  stacked = FALSE, do.stat = TRUE
) +
  scale_fill_manual(
    breaks = c("Rejection", "3month-protocol"), 
    values = explicit_colors
  ) +
  labs(fill = "Condition") +
  theme_classic(base_size = 12) +
  theme(panel.grid = element_blank(),
        axis.text.y = element_text(size = 9))

save_gg(p_unstacked, "03_rankNet",
        "RankNet_unstacked_contribution.pdf", w = 6, h = 8)

cat("Section 4 done.\n")

Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.


Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/03_rankNet/RankNet_stacked_contribution.pdf 


Scale for fill is already present.
Adding another scale for fill, which will replace the existing scale.


Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/03_rankNet/RankNet_unstacked_contribution.pdf 
Section 4 done.


# Pathway-level circle plots (for all shared pathways)

In [6]:
# ============================================================
# SECTION 5: Pathway-level circle plots (all shared pathways)
# (Supplement candidate)
# ============================================================

sub5 <- make_dir(BASE_OUT, "04_pathway_circles")

all_pathways_rej <- cc_rej@netP$pathways
all_pathways_3m  <- cc_3m@netP$pathways
all_pathways_all <- union(all_pathways_rej, all_pathways_3m)

for (pw in all_pathways_all) {
  f <- file.path(sub5, paste0(safe_name(pw), "_circle_comparison.pdf"))
  pdf(f, width = 14, height = 7)
  par(mfrow = c(1, 2))

  for (i in seq_along(object.list)) {
    obj   <- object.list[[i]]
    cname <- names(object.list)[i]
    if (pw %in% obj@netP$pathways) {
      netVisual_aggregate(obj, signaling = pw, layout = "circle",
                          signaling.name = paste(pw, cname))
      title(cname, line = 1, cex.main = 1.6, font.main = 2)
    } else {
      plot.new()
      title(paste0(cname, "\n(not detected)"), cex.main = 1.4)
    }
  }
  dev.off()
}
cat("Section 5 done. Circle plots:", length(all_pathways_all), "pathways.\n")


Section 5 done. Circle plots: 41 pathways.


# Chord diagrams — cell-level per pathway

In [7]:
# ============================================================
# SECTION 6: Chord diagrams — cell-level per pathway
# (Main or Supplement candidate)
# ============================================================

sub6 <- make_dir(BASE_OUT, "05_chord_diagrams")

for (pw in all_pathways_all) {
  f <- file.path(sub6, paste0(safe_name(pw), "_chord_comparison.pdf"))
  
  # 1. Widen the PDF to give the circular plots more breathing room
  pdf(f, width = 16, height = 8, useDingbats = FALSE)
  
  # 2. Adjust margins: c(bottom, left, top, right). 
  # The '6' on the right gives the legend its own dedicated space.
  # xpd = TRUE allows the legend to draw safely outside the plot boundaries.
  par(mfrow = c(1, 2), mar = c(2, 2, 4, 6), xpd = TRUE)

  for (i in seq_along(object.list)) {
    obj   <- object.list[[i]]
    cname <- names(object.list)[i]
    
    if (pw %in% obj@netP$pathways) {
      tryCatch({
        lr.use <- extractEnrichedLR(obj, signaling = pw, geneLR.return = FALSE)
        netVisual_chord_gene(obj, pairLR.use = lr.use,
                             title.name = paste0(pw, " — ", cname),
                             small.gap = 1, big.gap = 5, lab.cex = 0.8)
      }, error = function(e) {
        plot.new()
        title(paste0(pw, " — ", cname, "\n(plot error)"))
      })
    } else {
      plot.new()
      title(paste0(pw, " — ", cname, "\n(not detected)"))
    }
  }
  dev.off()
  
  # Safely clear the active graphics device to avoid Jupyter rendering crashes
  while (dev.cur() > 1) dev.off()
}

cat("Section 6 done.\n")

You may try the function `netVisual_chord_cell` for visualizing individual signaling pathway

You may try the function `netVisual_chord_cell` for visualizing individual signaling pathway

You may try the function `netVisual_chord_cell` for visualizing individual signaling pathway

You may try the function `netVisual_chord_cell` for visualizing individual signaling pathway

You may try the function `netVisual_chord_cell` for visualizing individual signaling pathway

You may try the function `netVisual_chord_cell` for visualizing individual signaling pathway

You may try the function `netVisual_chord_cell` for visualizing individual signaling pathway

You may try the function `netVisual_chord_cell` for visualizing individual signaling pathway

You may try the function `netVisual_chord_cell` for visualizing individual signaling pathway

You may try the function `netVisual_chord_cell` for visualizing individual signaling pathway

You may try the function `netVisual_chord_cell` for visualiz

Section 6 done.


# Bubble plots

In [8]:
# ============================================================
# SECTION 7: Bubble plots — all senders, both conditions
# (Supplement candidate)
# ============================================================

sub7 <- make_dir(BASE_OUT, "06_bubble_plots")

all_senders <- levels(cc_rej@idents)

for (sender in all_senders) {
  sname <- safe_name(sender)

  # Rejection
  f_rej <- file.path(sub7, paste0("Rejection_", sname, "_bubble.pdf"))
  pdf(f_rej, width = 12, height = 10)
  tryCatch(
    print(netVisual_bubble(cc_rej, sources.use = sender,
                           remove.isolate = FALSE,
                           title.name = paste("Rejection:", sender),
                           font.size = 11, angle.x = 90)),
    error = function(e) { plot.new(); title(paste(sender, "— no data")) }
  )
  dev.off()

  # 3month
  f_3m <- file.path(sub7, paste0("3month_", sname, "_bubble.pdf"))
  pdf(f_3m, width = 12, height = 10)
  tryCatch(
    print(netVisual_bubble(cc_3m, sources.use = sender,
                           remove.isolate = FALSE,
                           title.name = paste("3month-protocol:", sender),
                           font.size = 11, angle.x = 90)),
    error = function(e) { plot.new(); title(paste(sender, "— no data")) }
  )
  dev.off()
}
cat("Section 7 done.\n")


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 


Comparing communications on a single object 




Section 7 done.


# Count of unique pathways as sender and receiver

In [9]:
# ============================================================
# SECTION 8: Per-sender differential barplots
# (Count of unique pathways as sender and receiver)
# (Main Figure candidate: per cell type panel)
# ============================================================

sub8 <- make_dir(BASE_OUT, "07_sender_receiver_pathway_counts")

df_rej$group <- "Rejection"
df_3m$group  <- "3month-protocol"
df_combined  <- bind_rows(df_rej, df_3m) %>%
  mutate(group = factor(group, levels = COND_ORDER))

count_role <- function(df, role = c("sender", "receiver")) {
  role <- match.arg(role)
  col  <- if (role == "sender") "source" else "target"
  df %>%
    group_by(group, cell = .data[[col]]) %>%
    summarise(n_pathways = n_distinct(pathway_name), .groups = "drop")
}

sender_counts   <- count_role(df_combined, "sender")
receiver_counts <- count_role(df_combined, "receiver")

plot_role_bars <- function(data, role_label) {
  ggplot(data, aes(x = group, y = n_pathways, fill = group)) +
    geom_col(width = 0.6) +
    geom_text(aes(label = n_pathways), vjust = -0.4, size = 3.5) +
    facet_wrap(~cell, scales = "free_y") +
    scale_fill_manual(values = COND_COLORS) +
    scale_x_discrete(limits = COND_ORDER) +
    labs(title = paste("Unique pathways per cell type as", role_label),
         x = NULL, y = "# unique signaling pathways", fill = "Condition") +
    theme_classic(base_size = 11) +
    theme(axis.text.x = element_text(angle = 30, hjust = 1, size = 8),
          strip.text = element_text(size = 8, face = "bold"),
          legend.position = "top",
          plot.title = element_text(hjust = 0.5, face = "bold"))
}

p_sender_all   <- plot_role_bars(sender_counts,   "sender")
p_receiver_all <- plot_role_bars(receiver_counts, "receiver")

save_gg(p_sender_all,   "07_sender_receiver_pathway_counts",
        "All_celltypes_SENDER_pathway_counts.pdf",   w = 14, h = 10)
save_gg(p_receiver_all, "07_sender_receiver_pathway_counts",
        "All_celltypes_RECEIVER_pathway_counts.pdf", w = 14, h = 10)

cat("Section 8 done.\n")



Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/07_sender_receiver_pathway_counts/All_celltypes_SENDER_pathway_counts.pdf 
Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/07_sender_receiver_pathway_counts/All_celltypes_RECEIVER_pathway_counts.pdf 
Section 8 done.


# Pathway-specific sender-receiver CSVs

In [6]:
# ============================================================
# SECTION 9: Pathway-specific sender-receiver CSVs
# ============================================================

sub9 <- make_dir(BASE_OUT, "08_pathway_specific_tables")

all_pw_combined <- unique(c(df_rej$pathway_name, df_3m$pathway_name))

for (pw in all_pw_combined) {
  pw_data <- df_combined %>%
    filter(pathway_name == pw) %>%
    select(group, pathway_name, ligand, receptor,
           interaction_name_2, source, target, prob, pval) %>%
    arrange(group, source, target)
  if (nrow(pw_data) > 0) {
    save_csv(pw_data, "08_pathway_specific_tables",
             paste0(safe_name(pw), "_group_comparison.csv"))
  }
}

cat("Section 9 done.\n")

ERROR: Error: Objekt 'df_combined' nicht gefunden


# Communication pattern analysis

In [6]:
# ============================================================
# SECTION 10: Communication pattern analysis
# (Outgoing and incoming signaling patterns)
# (Supplement / Main candidate)
# ============================================================

sub10 <- make_dir(BASE_OUT, "09_communication_patterns")

# FIX: Explicitly load NMF to prevent the "keines der Pakete ist geladen" crash
suppressMessages(library(NMF))

for (i in seq_along(object.list)) {
  obj   <- object.list[[i]]
  cname <- names(object.list)[i]
  sname <- safe_name(cname)

  # -------------------------------------------------------------
  # 1. selectK: Cophenetic & Silhouette metrics to evaluate K
  # -------------------------------------------------------------
  
  # Outgoing selectK
  tryCatch({
    p_sel_out <- selectK(obj, pattern = "outgoing")

    save_gg(p_sel_out, "09_communication_patterns", paste0(sname, "_outgoing_selectK.pdf"), w = 10, h = 6)
  }, error = function(e) {
    message("outgoing selectK failed for ", cname, ": ", e$message)
  })

  # Incoming selectK
  tryCatch({
    p_sel_in <- selectK(obj, pattern = "incoming")
    save_gg(p_sel_in, "09_communication_patterns", paste0(sname, "_incoming_selectK.pdf"), w = 10, h = 6)
  }, error = function(e) {
    message("incoming selectK failed for ", cname, ": ", e$message)
  })

  # -------------------------------------------------------------
  # 2. identifyCommunicationPatterns
  # -------------------------------------------------------------
  k_use <- 4

  # Outgoing Patterns
  f_out_pattern <- file.path(sub10, paste0(sname, "_outgoing_patterns_k", k_use, ".pdf"))
  
  # Aggressively clear devices before opening the PDF canvas
  while (dev.cur() > 1) dev.off()
  
  pdf(f_out_pattern, width = 14, height = 18, useDingbats = FALSE)
  tryCatch({
    # identifyCommunicationPatterns returns the modified CellChat object 
    # but draws the ComplexHeatmap as a side-effect.
    obj <- identifyCommunicationPatterns(
      obj, pattern = "outgoing", k = k_use,
      title = paste(cname, "– outgoing"),
      width = 5, height = 14, font.size = 11
    )
  }, error = function(e) {
    plot.new()
    title(paste(cname, "outgoing pattern failed:\n", e$message), cex.main = 0.8)
  })
  dev.off()

  # Incoming Patterns
  f_in_pattern <- file.path(sub10, paste0(sname, "_incoming_patterns_k", k_use, ".pdf"))
  
  while (dev.cur() > 1) dev.off()
  
  pdf(f_in_pattern, width = 14, height = 18, useDingbats = FALSE)
  tryCatch({
    obj <- identifyCommunicationPatterns(
      obj, pattern = "incoming", k = k_use,
      title = paste(cname, "– incoming"),
      width = 5, height = 14, font.size = 11
    )
  }, error = function(e) {
    plot.new()
    title(paste(cname, "incoming pattern failed:\n", e$message), cex.main = 0.8)
  })
  dev.off()
}

# Final cleanup of the graphics device
while (dev.cur() > 1) dev.off()

cat("Section 10 done.\n")

Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/09_communication_patterns/Rejection_outgoing_selectK.pdf 


Zeitnahme beendet um: 7.939 0.715 8.603



# Signaling role scatter plot (2D centrality)

In [11]:
# ============================================================
# SECTION 11: Signaling role scatter (2D centrality)
# (Main Figure candidate: panel C or D)
# Shows which cell types are dominant senders vs. receivers
# ============================================================

sub11 <- make_dir(BASE_OUT, "10_signaling_roles")

for (i in seq_along(object.list)) {
  obj   <- object.list[[i]]
  cname <- names(object.list)[i]
  sname <- safe_name(cname)

  f <- file.path(sub11, paste0(sname, "_signaling_role_2D.pdf"))
  pdf(f, width = 8, height = 6)
  tryCatch({
    p <- netAnalysis_signalingRole_scatter(obj,
           title = paste(cname, "– signaling role"))
    print(p)
  }, error = function(e) {
    plot.new()
    title(paste(cname, "– role scatter failed"))
  })
  dev.off()
}

# Side-by-side comparison
pdf(file.path(sub11, "Both_conditions_signaling_role_2D.pdf"),
    width = 14, height = 6)
par(mfrow = c(1, 2))
for (i in seq_along(object.list)) {
  obj   <- object.list[[i]]
  cname <- names(object.list)[i]
  tryCatch(
    print(netAnalysis_signalingRole_scatter(obj, title = cname)),
    error = function(e) { plot.new(); title(paste(cname, "– failed")) }
  )
}
dev.off()

cat("Section 11 done.\n")


Signaling role analysis on the aggregated cell-cell communication network from all signaling pathways

Signaling role analysis on the aggregated cell-cell communication network from all signaling pathways

Signaling role analysis on the aggregated cell-cell communication network from all signaling pathways

Signaling role analysis on the aggregated cell-cell communication network from all signaling pathways



agg_record_117343332 
                   2

Section 11 done.


# Heatmaps of outgoing/incoming signaling strength

In [12]:
# ============================================================
# SECTION 12: Heatmaps of outgoing/incoming signaling strength
# per cell type per pathway
# (Main Figure candidate: panel E/F)
# ============================================================

sub12 <- make_dir(BASE_OUT, "11_signaling_heatmaps")

for (i in seq_along(object.list)) {
  obj   <- object.list[[i]]
  cname <- names(object.list)[i]
  sname <- safe_name(cname)

  # 1. Outgoing Signaling Heatmap
  f_out <- file.path(sub12, paste0(sname, "_outgoing_heatmap.pdf"))
  
  # Ensure clean slate
  while (dev.cur() > 1) dev.off()
  pdf(f_out, width = 10, height = 12, useDingbats = FALSE)
  
  tryCatch({
    # FIX: Wrapped in print() to force the ComplexHeatmap object to render
    print(netAnalysis_signalingRole_heatmap(obj, pattern = "outgoing",
                                            title = paste(cname, "– outgoing signaling")))
  }, error = function(e) {
    plot.new()
    title(paste("outgoing heatmap failed:", e$message), cex.main = 0.8)
  })
  dev.off()

  # 2. Incoming Signaling Heatmap
  f_in <- file.path(sub12, paste0(sname, "_incoming_heatmap.pdf"))
  
  # Ensure clean slate
  while (dev.cur() > 1) dev.off()
  pdf(f_in, width = 10, height = 10, useDingbats = FALSE)
  
  tryCatch({
    # FIX: Wrapped in print() to force the ComplexHeatmap object to render
    print(netAnalysis_signalingRole_heatmap(obj, pattern = "incoming",
                                            title = paste(cname, "– incoming signaling")))
  }, error = function(e) {
    plot.new()
    title(paste("incoming heatmap failed:", e$message), cex.main = 0.8)
  })
  dev.off()
}

# Final cleanup of the graphics device
while (dev.cur() > 1) dev.off()

cat("Section 12 done.\n")

Section 12 done.


# Master interaction comparison CSV (All senders × pathways × conditions — wide format)

In [13]:
# ============================================================
# SECTION 13: Master interaction comparison CSV
# (All senders × pathways × conditions — wide format)
# ============================================================

edges_df <- df_combined %>%
  group_by(group, source, pathway_name, target) %>%
  summarise(
    total_prob   = sum(prob, na.rm = TRUE),
    min_pval     = suppressWarnings(min(pval, na.rm = TRUE)),
    .groups = "drop"
  )

wide_edges <- edges_df %>%
  pivot_wider(
    id_cols     = c(source, pathway_name, target),
    names_from  = group,
    values_from = c(total_prob, min_pval),
    values_fill = list(total_prob = 0, min_pval = NA)
  ) %>%
  rename_with(~ gsub("-", "_", .x)) %>%
  mutate(
    diff_prob = total_prob_Rejection - `total_prob_3month_protocol`,
    log2fc    = log2((total_prob_Rejection + 1e-6) /
                     (`total_prob_3month_protocol` + 1e-6))
  ) %>%
  arrange(source, pathway_name, desc(diff_prob))

save_csv(wide_edges, "01_tables", "Master_sender_pathway_target_comparison.csv")

cat("Section 13 done.\n")



Saved: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/01_tables/Master_sender_pathway_target_comparison.csv 
Section 13 done.


# Differential circle plots

In [3]:
# ============================================================
# SECTION 13A: Per-sender differential circle plots
# Rejection higher = salmon (#E09E9E)
# 3month higher    = skyblue
# ============================================================

library(CellChat)

# ── FIX 6 Oct 2026 ─────────────────────────────────────────────────────────────────────────────────────
# CellChat 1.6.1 aggregateNet(signaling = ...) pastes source and target with "_" and splits them again, which breaks
# every group whose name contains "_" (EC_papio edges were merged into EC; edges to/from Mye_sus, Mye_papio,
# Lymph_papio and Lymph_sus were dropped or mis-assigned, and some plots failed). aggregateNet_fixed() below has the
# same semantics (same thresh, count = significant L-R pairs, weight = summed probability) but aggregates source and
# target directly. Labels are unchanged. Audit of old vs corrected edges: 12_comparative_chord_plots/diffcircle_fix_audit.csv
# Drop-in replacement for CellChat 1.6.1 aggregateNet(object, signaling = ...): identical semantics
# (subsetCommunication with the same thresh, count = number of significant L-R pairs, weight = summed
# probability), but sources and targets are aggregated directly instead of being pasted with "_" and
# split again - the split broke every group name that contains "_" (Mye_sus, EC_papio, ...).
aggregateNet_fixed <- function(object, signaling, remove.isolate = FALSE, thresh = 0.05) {
  net <- object@net
  df.net <- subsetCommunication(object, slot.name = "net", signaling = signaling, thresh = thresh)
  cells.level <- levels(object@idents)
  src <- factor(as.character(df.net$source), levels = cells.level)
  tgt <- factor(as.character(df.net$target), levels = cells.level)
  if (remove.isolate) {
    src <- droplevels(src); tgt <- droplevels(tgt)
  }
  count <- tapply(rep(1, nrow(df.net)), list(src, tgt), sum)
  prob  <- tapply(df.net$prob, list(src, tgt), sum)
  count[is.na(count)] <- 0; prob[is.na(prob)] <- 0
  net$count <- count; net$weight <- prob
  object@net <- net
  object
}

# Drop-in copy of CellChat 1.6.1 netVisual_diffInteraction with ONE added line: every edge gets loop.angle = 0
# before the self-loop angles are written. Without it, igraph refuses the partial assignment
# ("Length of new attribute value must be 1 or <n>") whenever the sender also signals to itself, and the plot fails.
.nvd_src <- deparse(CellChat::netVisual_diffInteraction)
.nvd_i <- grep("if (sum(edge.start[, 2] == edge.start[, 1]) != 0)", .nvd_src, fixed = TRUE)[1]
stopifnot(!is.na(.nvd_i))
.nvd_src <- append(.nvd_src, "    igraph::E(g)$loop.angle <- 0", after = .nvd_i - 1)
netVisual_diffInteraction_fixed <- eval(parse(text = .nvd_src))
environment(netVisual_diffInteraction_fixed) <- asNamespace("CellChat")



safe_fn <- function(x) gsub("[/\\\\ ]", "_", x)

all_senders       <- levels(cc_rej@idents)
master_dir        <- make_dir(BASE_OUT, "12_comparative_chord_plots")
weight_max_weight <- getMaxWeight(object.list, attribute = "weight")

for (my_sender in all_senders) {

  cat("--- Processing sender:", my_sender, "---\n")
  safe_sender <- safe_fn(my_sender)
  sender_dir  <- make_dir(master_dir, safe_sender)

  # Get pathways to plot from rankNet
  ranknet_frame <- tryCatch({
    rankNet(
      cellchat_merged,
      mode          = "comparison",
      stacked       = TRUE,
      do.stat       = TRUE,
      cutoff.pvalue = 0.05,
      return.data   = TRUE,
      sources.use   = my_sender,
      measure       = "weight"
    )$signaling.contribution
  }, error = function(e) {
    message("rankNet failed for: ", my_sender, " — ", e$message)
    NULL
  })

  if (is.null(ranknet_frame) || nrow(ranknet_frame) == 0) {
    cat("No rankNet data for:", my_sender, "— skipping.\n")
    next
  }

  write.csv(ranknet_frame,
            file.path(sender_dir, paste0(safe_sender, "_rankNet_diff_table.csv")),
            row.names = FALSE)

  pw_to_plot <- unique(ranknet_frame$name)

  for (this_pathway in pw_to_plot) {

    in_rej <- this_pathway %in% cc_rej@netP$pathways
    in_3m  <- this_pathway %in% cc_3m@netP$pathways

    if (!in_rej && !in_3m) next

    # Verify direction from netP — same source as rankNet
    prob_rej <- tryCatch({
      if (in_rej) {
        mat <- cc_rej@netP$prob[, , this_pathway]
        if (my_sender %in% rownames(mat)) sum(mat[my_sender, ], na.rm = TRUE) else 0
      } else 0
    }, error = function(e) 0)

    prob_3m <- tryCatch({
      if (in_3m) {
        mat <- cc_3m@netP$prob[, , this_pathway]
        if (my_sender %in% rownames(mat)) sum(mat[my_sender, ], na.rm = TRUE) else 0
      } else 0
    }, error = function(e) 0)

    higher_in <- ifelse(prob_rej >= prob_3m, "Rejection", "3month-protocol")

    group_rej <- tryCatch(
      aggregateNet_fixed(cc_rej, signaling = this_pathway, remove.isolate = FALSE),
      error = function(e) NULL
    )
    group_3m <- tryCatch(
      aggregateNet_fixed(cc_3m, signaling = this_pathway, remove.isolate = FALSE),
      error = function(e) NULL
    )

    if (is.null(group_rej) || is.null(group_3m)) {
      message("  Skipping ", this_pathway, ": aggregateNet failed")
      next
    }

    # [[1]] = 3month (obj1), [[2]] = Rejection (obj2)
    # net.diff = Rejection - 3month
    # positive (Rej > 3mo) → color.edge[1] = salmon ✅
    # negative (3mo > Rej) → color.edge[2] = skyblue ✅
    pathway_list <- list(`3month-protocol` = group_3m,
                          Rejection         = group_rej)

    cc_pw <- tryCatch(
      mergeCellChat(pathway_list, add.names = names(pathway_list)),
      error = function(e) {
        message("  mergeCellChat failed: ", e$message); NULL
      }
    )
    if (is.null(cc_pw)) next

    weight_max_pw <- tryCatch(
      getMaxWeight(pathway_list, slot.name = "netP", attribute = this_pathway),
      error = function(e) NULL
    )
    if (is.null(weight_max_pw)) next

    safe_pw   <- safe_fn(this_pathway)
    f_out     <- file.path(sender_dir,
                           paste0(safe_pw, "_diffcircle_", safe_sender, ".pdf"))
    title_str <- sprintf(
      "%s | sender: %s\nRej=%.4f  3mo=%.4f  [Higher: %s]",
      this_pathway, my_sender, prob_rej, prob_3m, higher_in
    )

    while (dev.cur() > 1) dev.off()

    plot_ok <- tryCatch({
      pdf(f_out, width = 7, height = 8, useDingbats = FALSE)
      par(mar = c(1, 1, 5, 1))
      netVisual_diffInteraction_fixed(
        cc_pw,
        weight.scale     = TRUE,
        measure          = "weight",
        comparison       = c(1, 2),
        color.edge       = c("#E09E9E", "skyblue"),
        title.name       = title_str,
        label.edge       = FALSE,
        arrow.size       = 1.5,
        arrow.width      = 2,
        alpha.edge       = 0.9,
        margin           = 0.4,
        vertex.label.cex = 1.1,
        edge.weight.max  = weight_max_pw[1],
        edge.width.max   = 10,
        sources.use      = my_sender
      )
      TRUE
    }, error = function(e) {
      message("  Plot failed for ", this_pathway, ": ", e$message)
      FALSE
    })

    while (dev.cur() > 1) dev.off()

    if (!plot_ok && file.exists(f_out)) file.remove(f_out)

    cat("  ", this_pathway,
        "| Rej:", round(prob_rej, 4),
        "| 3mo:", round(prob_3m, 4),
        "| Higher:", higher_in, "\n")
  }

  cat("Finished sender:", my_sender, "\n\n")
}

while (dev.cur() > 1) dev.off()
cat("Section 13A complete. Results in:", master_dir, "\n")
invisible(NULL)

--- Processing sender: CM ---


Merge the following slots: 'data.signaling','images','net', 'netP','meta', 'idents', 'var.features' , 'DB', and 'LR'.



   FGF | Rej: 0.0254 | 3mo: 0.0511 | Higher: 3month-protocol 


Merge the following slots: 'data.signaling','images','net', 'netP','meta', 'idents', 'var.features' , 'DB', and 'LR'.



   APP | Rej: 0.0827 | 3mo: 0.1054 | Higher: 3month-protocol 


Merge the following slots: 'data.signaling','images','net', 'netP','meta', 'idents', 'var.features' , 'DB', and 'LR'.



... (178 outputs left out)

  Skipping HSPG: aggregateNet failed



Merge the following slots: 'data.signaling','images','net', 'netP','meta', 'idents', 'var.features' , 'DB', and 'LR'.



   CSF | Rej: 0.0294 | 3mo: 0 | Higher: Rejection 
Finished sender: NC 



Section 13A complete. Results in: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/12_comparative_chord_plots 


# Saving cell type as sender ranknet plots + CSV

In [7]:
# ============================================================
# SECTION 13B: Per-sender RankNet stacked bar plots (standalone)
# Rejection = salmon (#E09E9E), 3month-protocol = skyblue
# Colors mapped by condition name — robust to ggplot internals
# ============================================================

library(CellChat)
library(ggplot2)

safe_fn    <- function(x) gsub("[/\\\\ ]", "_", x)
all_senders <- levels(cc_rej@idents)
master_dir  <- make_dir(BASE_OUT, "12_comparative_chord_plots")

# Fixed color map by condition name — does not depend on ggplot_build
FIXED_COLORS <- c(
  "Rejection"       = "#E09E9E",
  "3month-protocol" = "skyblue"
)

for (my_sender in all_senders) {

  cat("--- RankNet bar for sender:", my_sender, "---\n")
  safe_sender <- safe_fn(my_sender)
  sender_dir  <- make_dir(master_dir, safe_sender)

  # Get the raw rankNet data for this sender
  ranknet_data <- tryCatch({
    rankNet(
      cellchat_merged,
      mode          = "comparison",
      stacked       = TRUE,
      do.stat       = TRUE,
      cutoff.pvalue = 0.05,
      return.data   = TRUE,
      sources.use   = my_sender,
      measure       = "weight"
    )$signaling.contribution
  }, error = function(e) {
    message("rankNet data failed for: ", my_sender)
    NULL
  })

  if (is.null(ranknet_data) || nrow(ranknet_data) == 0) {
    cat("  No data — skipping.\n")
    next
  }

  # Build the plot manually from the data frame
  # This completely bypasses ggplot_build color detection issues
  # ranknet_data columns: name, contribution.scaled, group, pvalues
  # 'group' contains the condition names directly

  # Order pathways by total contribution for clean visualization
  pathway_order <- ranknet_data %>%
    group_by(name) %>%
    summarise(total = sum(contribution.scaled, na.rm = TRUE),
              .groups = "drop") %>%
    arrange(total) %>%
    pull(name)

  ranknet_data$name  <- factor(ranknet_data$name,  levels = pathway_order)
  ranknet_data$group <- factor(ranknet_data$group,
                                levels = c("Rejection", "3month-protocol"))

  p_rank <- ggplot(ranknet_data,
                   aes(x = contribution.scaled, y = name, fill = group)) +
    geom_col(position = "fill") +
    geom_vline(xintercept = 0.5, linetype = "dashed",
               color = "grey40", linewidth = 0.4) +
    scale_fill_manual(
      values = FIXED_COLORS,
      labels = c("Rejection", "3month-protocol")
    ) +
    scale_x_continuous(labels = scales::label_number(accuracy = 0.01)) +
    labs(
      title = paste("Differential signaling —", my_sender),
      x     = "Relative information flow",
      y     = NULL,
      fill  = "Condition"
    ) +
    theme_classic(base_size = 12) +
    theme(
      plot.title      = element_text(hjust = 0.5, face = "bold"),
      panel.grid      = element_blank(),
      legend.position = "top",
      axis.text.y     = element_text(size = 9),
      axis.text.x     = element_text(size = 9)
    )

  ggsave(
    file.path(sender_dir, paste0(safe_sender, "_rankNet_stacked.pdf")),
    plot   = p_rank,
    device = "pdf",
    width  = 2, height = max(4, 0.2 * length(unique(ranknet_data$name)) + 0.75),
    units  = "in",
    dpi    = 300,
    useDingbats = FALSE
  )
  cat("  Saved:", safe_sender, "_rankNet_stacked.pdf\n")
}

cat("Section 13B complete.\n")
invisible(NULL)

--- RankNet bar for sender: CM ---
  Saved: CM _rankNet_stacked.pdf
--- RankNet bar for sender: EC ---
  Saved: EC _rankNet_stacked.pdf
--- RankNet bar for sender: EC_papio ---
  Saved: EC_papio _rankNet_stacked.pdf
--- RankNet bar for sender: FB ---
  Saved: FB _rankNet_stacked.pdf
--- RankNet bar for sender: Lymph_papio ---
  Saved: Lymph_papio _rankNet_stacked.pdf
--- RankNet bar for sender: Lymph_sus ---


rankNet data failed for: Lymph_sus



  No data — skipping.
--- RankNet bar for sender: MC ---
  Saved: MC _rankNet_stacked.pdf
--- RankNet bar for sender: Mye_papio ---
  Saved: Mye_papio _rankNet_stacked.pdf
--- RankNet bar for sender: Mye_sus ---
  Saved: Mye_sus _rankNet_stacked.pdf
--- RankNet bar for sender: NC ---
  Saved: NC _rankNet_stacked.pdf
Section 13B complete.


# Custom fancy chord diagrams (circilize)

In [2]:
# ============================================================
# SECTION 14: Custom Fancy Chord Diagrams (circlize)
# ============================================================

suppressMessages({
  library(dplyr)
  library(tidyr)
  library(circlize)
})

# 1. Setup Directories and Load Master Data
sub14 <- make_dir(BASE_OUT, "13_Fancy_Chord_Diagrams")

# FIX: Changed filenames to match exactly what was exported in Section 2
df_rej <- read.csv(file.path(BASE_OUT, "01_tables", "CellChat_Rejection_all_interactions.csv"))
df_3m  <- read.csv(file.path(BASE_OUT, "01_tables", "CellChat_3month_all_interactions.csv"))

# Store in a named list to easily loop through them
conditions_data <- list(
  "Rejection" = df_rej,
  "3month-protocol" = df_3m
)

# 2. Define Parameters
pathways_to_plot <- c("VEGF", "ANGPT", "CCL", "VISFATIN", "HSPG", "PERIOSTIN")   # PERIOSTIN = POSTN, added 6 Oct 2026
# circlize 0.4.18 silently drops the FB->CM link of PERIOSTIN when "diffHeight" and "big.arrow" are combined
# (verified 6 Oct 2026 by counting drawn links: 1 of 2). For the pathways listed here the arrows are drawn without
# diffHeight, which draws every link; all other pathways keep the original style (all their links verified drawn).
ARROWS_ONLY <- c("PERIOSTIN")

custom_colors <- c(
    "CM" =          "#4FBDBA",
    "FB" =          "#FADA5E",
    "EC" =          "#5499C7",
    "EC_papio" =    "#B0E0E6",
    "MC" =          "#FF8C7A",
    "Mye_papio" =   "#94708E",
    "Mye_sus" =     "#C9A0DC",
    "Lymph_papio" = "#D2691E",
    "Lymph_sus" =   "#FFCBA4",
    "NC" =          "#1B5E20"
)

# 3. Loop through Conditions and Pathways to Generate Plots
for (cond_name in names(conditions_data)) {
  df_current <- conditions_data[[cond_name]]
  
  for (pw in pathways_to_plot) {
    
    # Filter for the specific pathway
    pw_data <- df_current %>% filter(pathway_name == pw)
    
    # Skip if the pathway doesn't exist in this condition
    if (nrow(pw_data) == 0) {
      cat(sprintf("Skipping %s in %s: No interactions found.\n", pw, cond_name))
      next
    }
    
    # Prepare the output filename
    safe_pw <- safe_name(pw)
    safe_cond <- safe_name(cond_name)
    pdf_file <- file.path(sub14, sprintf("%s_circos_%s.pdf", safe_pw, safe_cond))
    
    # Ensure a clean graphics device
    while (dev.cur() > 1) dev.off()
    
    # Start PDF
    pdf(file = pdf_file, width = 7, height = 7, useDingbats = FALSE)
    
    tryCatch({
      # Reset circlize parameters
      circos.clear()
      circos.par(start.degree = 90, gap.degree = 15)
      
      # Generate the chord diagram
      chordDiagram(
        x = pw_data %>% select(source, target, prob),
        grid.col = custom_colors,
        directional = 1,
        direction.type = if (pw %in% ARROWS_ONLY) "arrows" else c("diffHeight", "arrows"),
        link.arr.type = "big.arrow",
        link.arr.length = 0.15,
        link.sort = TRUE,
        link.border = "grey"
      )
      
      # Add external labels
      circos.track(track.index = 1, ylim = c(0, 1), panel.fun = function(x, y) {
        circos.text(CELL_META$xcenter, CELL_META$ylim[1] + 1.5, CELL_META$sector.index,
                    facing = "clockwise", niceFacing = TRUE, adj = c(0, 0.5), cex = 1.0)
      }, bg.border = NA)
      
      # Add Title
      title(sprintf("%s Signaling Interactions in %s", pw, cond_name))
      
      cat(sprintf("Saved: %s circos plot for %s\n", pw, cond_name))
      
    }, error = function(e) {
      plot.new()
      title(sprintf("Failed to plot %s for %s:\n%s", pw, cond_name, e$message), cex.main = 0.8)
    })
    
    dev.off()
    
    # Always clear the circlize environment state after plotting
    circos.clear()
  }
}

while (dev.cur() > 1) dev.off()
cat("Section 14 complete. Fancy circos plots are in:", sub14, "\n")

Note: 1 point is out of plotting region in sector 'CM', track '1'.



Note: 1 point is out of plotting region in sector 'CM', track '1'.



Note: 1 point is out of plotting region in sector 'EC', track '1'.



Note: 1 point is out of plotting region in sector 'EC', track '1'.



Note: 1 point is out of plotting region in sector 'EC_papio', track
'1'.



Note: 1 point is out of plotting region in sector 'EC_papio', track
'1'.



... (46 outputs left out)

Note: 1 point is out of plotting region in sector 'EC_papio', track
'1'.



Note: 1 point is out of plotting region in sector 'EC_papio', track
'1'.



Saved: VEGF circos plot for 3month-protocol
Skipping ANGPT in 3month-protocol: No interactions found.
Skipping CCL in 3month-protocol: No interactions found.
Skipping VISFATIN in 3month-protocol: No interactions found.
Skipping HSPG in 3month-protocol: No interactions found.
Skipping PERIOSTIN in 3month-protocol: No interactions found.


Section 14 complete. Fancy circos plots are in: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/13_Fancy_Chord_Diagrams 


# Extra analysis: sender-target heatmap per cell_type

In [17]:
# ============================================================
# SECTION 16: Sender-Target Communication Counts Heatmaps
# (Fixed Scale: 0 to 40 | Palette: Blue -> Yellow -> Red)
# ============================================================

suppressMessages({
  library(ComplexHeatmap) 
  library(circlize) 
  library(dplyr)
  library(tidyr)
})

# 1. Setup output directory
sub16 <- make_dir(BASE_OUT, "15_Sender_Target_Heatmaps")

# 2. Point to the CSV files generated in Section 2
csv_files <- list(
  "Rejection" = file.path(BASE_OUT, "01_tables", "CellChat_Rejection_all_interactions.csv"),
  "3month-protocol" = file.path(BASE_OUT, "01_tables", "CellChat_3month_all_interactions.csv")
)

# 3. Define universal color scale BEFORE the loop
# 0 = Dark Blue, 20 = Light Yellow, 40 = Dark Red
universal_color_mapping <- colorRamp2(
  c(0, 20, 40), 
  c("#313695", "#ffffbf", "#a50026") 
)

# 4. Loop through conditions to build matrices and plot heatmaps
for (cond_name in names(csv_files)) {
  
  csv_path <- csv_files[[cond_name]]
  
  if (!file.exists(csv_path)) {
    cat("File not found:", csv_path, "\n")
    next
  }
  
  df_cond <- read.csv(csv_path, stringsAsFactors = FALSE)
  
  # Build sender–target count matrix for p < 0.05
  heat <- df_cond %>%
    mutate(pval = as.numeric(as.character(pval))) %>%
    filter(pval < 0.05) %>%
    group_by(source, target) %>%
    summarize(count = n(), .groups = "drop") %>%
    pivot_wider(names_from = target, values_from = count, values_fill = 0)
  
  # Check if data exists after filtering
  if (nrow(heat) == 0) {
    cat("No significant interactions (p < 0.05) found for", cond_name, "- skipping.\n")
    next
  }
  
  # Convert to matrix
  mat <- as.matrix(heat[, -1])
  rownames(mat) <- heat$source
  
  # Define output filename and clean title
  safe_cond <- safe_name(cond_name)
  out_name  <- sprintf("SenderTargetHeatmap_%s.pdf", safe_cond)
  out_path  <- file.path(sub16, out_name)
  plot_title <- sprintf("Sender-target communication counts (%s)", cond_name)
  
  # Ensure clean graphics device
  while (dev.cur() > 1) dev.off()
  
  # Save the clustered heatmap
  pdf(out_path, width = 6, height = 5, useDingbats = FALSE)
  tryCatch({
    ht <- Heatmap(
      mat, 
      col = universal_color_mapping, 
      cluster_rows = TRUE, 
      cluster_columns = TRUE, 
      column_title = plot_title,
      name = "Count", # FIX: Simple string without \n to prevent grid viewport crash
      heatmap_legend_param = list(title = "Count\n(Max 40)"), # Safely puts the \n in the legend
      row_names_side = "left",
      column_names_rot = 45 
    )
    
    print(ht) 
    
    cat("Saved ComplexHeatmap for", cond_name, "to:", out_name, "\n")
  }, error = function(e) {
    plot.new()
    title(sprintf("Failed to plot heatmap for %s:\n%s", cond_name, e$message), cex.main = 0.8)
    cat("  -> Plot failed:", e$message, "\n")
  })
  dev.off()
}

# Final cleanup
while (dev.cur() > 1) dev.off()
cat("Section 16 complete.\n")

Saved ComplexHeatmap for Rejection to: SenderTargetHeatmap_Rejection.pdf 
Saved ComplexHeatmap for 3month-protocol to: SenderTargetHeatmap_3month-protocol.pdf 
Section 16 complete.


# FB: Cell-Type Specific IN/OUT Interaction Heatmap

In [18]:
# ============================================================
# SECTION 17: Cell-Type Specific IN/OUT Interaction Heatmap
# (Focus: Fibroblasts / FB)
# ============================================================

suppressMessages({
  library(ComplexHeatmap) # Bypassing pheatmap to prevent cluster errors
  library(circlize)
  library(dplyr)
})

# 1. Setup Data and Output Directory
sub17 <- make_dir(BASE_OUT, "16_CellType_Specific_Heatmaps")
out_pdf <- file.path(sub17, "Rejection_FB_Cell_Cell_Count_Ordered_Heatmap.pdf")

# Cell type of interest
cell_oi <- "FB"
conditions <- c("3month-protocol", "Rejection")

# Initialize list to store filtered count matrices
fibro_matrices <- list()

# --- DEFINE DESIRED CELL ORDER (Updated to short names) ---
cell_order_display <- c(
    "CM", "FB", "EC", "EC_papio", "MC", 
    "Mye_papio", "Mye_sus", "Lymph_papio", "Lymph_sus", "NC"
)
# ----------------------------------------------------------

# 2. Extract and format interaction counts
for (cond in conditions) {
    # Extract the object from the merged list
    cc_obj <- object.list[[cond]]
    
    # Get the full interaction count matrix
    full_count_matrix <- cc_obj@net$count
    
    # Identify all partner cells
    all_cells <- unique(c(rownames(full_count_matrix), colnames(full_count_matrix)))
    partners  <- setdiff(all_cells, cell_oi)
    
    # Safely extract sender/receiver counts (handling cases where FB might be missing)
    sent_by_fb <- if(cell_oi %in% rownames(full_count_matrix)) full_count_matrix[cell_oi, all_cells] else rep(0, length(all_cells))
    recv_by_fb <- if(cell_oi %in% colnames(full_count_matrix)) full_count_matrix[all_cells, cell_oi] else rep(0, length(all_cells))
    
    # Create comparative data frame
    df_comp <- data.frame(
        'Source' = all_cells, 
        'Fibro_as_Sender' = sent_by_fb, 
        'Fibro_as_Receiver' = recv_by_fb 
    ) %>%
        filter(Source != cell_oi) %>% # Remove FB-to-FB self-loops
        select(Source, Fibro_as_Sender, Fibro_as_Receiver)
    
    # Convert to matrices for heatmap
    sender_mat <- matrix(df_comp$Fibro_as_Sender, ncol = 1, dimnames = list(df_comp$Source, paste0(cond, "_Sent")))
    receiver_mat <- matrix(df_comp$Fibro_as_Receiver, ncol = 1, dimnames = list(df_comp$Source, paste0(cond, "_Received")))
    
    # Combine roles
    fibro_matrices[[cond]] <- cbind(receiver_mat, sender_mat)
}

# 3. Final Combined Matrix (Rows = Partner Cells, Columns = 4 roles)
final_mat_raw <- cbind(fibro_matrices[['3month-protocol']], fibro_matrices[['Rejection']])
colnames(final_mat_raw) <- c("Received (3month)", "Sent (3month)", "Received (Rejection)", "Sent (Rejection)")

valid_order_partners <- intersect(setdiff(cell_order_display, cell_oi), rownames(final_mat_raw))
final_mat <- final_mat_raw[valid_order_partners, , drop = FALSE]

# 4. ComplexHeatmap Visualization

# Define top annotations (Replaces pheatmap annotation_col)
column_ha <- HeatmapAnnotation(
    Condition = c("3month-protocol", "3month-protocol", "Rejection", "Rejection"),
    Role = c("Received by FB", "Sent by FB", "Received by FB", "Sent by FB"),
    col = list(
        Condition = c("Rejection" = "#08528f", "3month-protocol" = "skyblue"),
        Role = c("Received by FB" = "grey90", "Sent by FB" = "grey40")
    ),
    annotation_name_side = "left"
)

# Define gradient color mapping (White -> Orange -> Red)
max_val <- max(final_mat, na.rm = TRUE)
ht_colors <- colorRamp2(c(0, max_val/2, max_val), c("white", "orange", "red"))

# Ensure clean graphics device
while (dev.cur() > 1) dev.off()

# Save PDF
pdf(file = out_pdf, width = 6, height = 7, useDingbats = FALSE)

tryCatch({
    ht <- Heatmap(
        final_mat,
        col = ht_colors,
        cluster_rows = FALSE,   
        cluster_columns = FALSE, 
        top_annotation = column_ha,
        row_names_side = "left",
        column_title = paste0("Comparative pathway counts sent/received by ", cell_oi),
        name = "Count",
        
        # This replaces pheatmap's 'display_numbers = TRUE'
        cell_fun = function(j, i, x, y, width, height, fill) {
            grid.text(final_mat[i, j], x, y, gp = gpar(fontsize = 10, col = "black"))
        }
    )
    
    print(ht)
    cat("Saved comparative FB interaction count heatmap to:", out_pdf, "\n")
    
}, error = function(e) {
    plot.new()
    title(paste("Failed to plot FB Heatmap:\n", e$message), cex.main = 0.8)
})

dev.off()

# Clean up
while (dev.cur() > 1) dev.off()
cat("Section 17 complete.\n")

Saved comparative FB interaction count heatmap to: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/Diff_CC/Rej_vs_3month/results/16_CellType_Specific_Heatmaps/Rejection_FB_Cell_Cell_Count_Ordered_Heatmap.pdf 


png 
  2

Section 17 complete.
